# Supplier-Removal Simulation
 
**Purpose:**  

> *"If Russia stopped exporting wheat, N countries would lose more than half  
> their wheat supply."*

---

## What This Notebook Does

1. **Targeted removal:** Remove one major exporter (Russia for wheat, India for  
   rice, US for maize), recompute each importer's remaining supply, and report  
   the volume lost, share lost, and suppliers remaining.  
2. **Random removal baseline:** Remove a random exporter of equivalent rank to  
   show that targeted removal is consistently more damaging. This converts the  
   simulation from an anecdote into a structural finding.  
3. **Multi-exporter cascade:** Remove the top-2 and top-3 exporters together to  
   show compounding fragility.  
4. **Headline statistics and charts** ready for the presentation.

### Assumption (stated honestly)

This is a **static accounting shock** — no substitution, no price response, no  
rerouting. It is a worst-case bound, not a forecast. Real markets partially  
reroute, which is precisely why 2022 raised prices more than it cut volumes.

---

## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)

# ---- Paths ----
# ROOT resolves to the repo root assuming notebook lives in notebooks/
ROOT = Path(".").resolve().parent
TRADE_MATRIX = ROOT / "data" / "cleaned" / "trade_matrix_cleaned.csv"
VIZ_DIR = ROOT / "visualizations"
OUTPUT_DIR = ROOT / "data" / "cleaned"
VIZ_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ---- Item-to-commodity mapping (matches build_quantity_concentration.py) ----
ITEM_MAP = {
    "Wheat": "Wheat",
    "Wheat and meslin flour": "Wheat",
    "Rice, paddy (rice milled equivalent)": "Rice",
    "Rice, milled": "Rice",
    "Maize (corn)": "Maize",
}

# ---- Simulation parameters ----
# We use a 3-year mean (2021-2023) as the methodology spec recommends.
# 2021-2023 keeps one year of Russian self-reporting as a cross-check.
WINDOW_START = 2021
WINDOW_END = 2023

# The exporter to remove for each commodity.
# These are the dominant global exporters identified in the data audit.
TARGETED_EXPORTERS = {
    "Wheat": "Russian Federation",
    "Rice": "India",
    "Maize": "United States of America",
}

# For the cascade simulation: top exporters to remove sequentially.
CASCADE_EXPORTERS = {
    "Wheat": ["Russian Federation", "Australia", "Canada"],
    "Rice": ["India", "Thailand", "Viet Nam"],
    "Maize": ["United States of America", "Argentina", "Brazil"],
}

# Number of random removal trials for the baseline comparison.
N_RANDOM_TRIALS = 200

# Seed for reproducibility of random removal.
RNG_SEED = 42

print(f"Trade matrix: {TRADE_MATRIX}")
print(f"Exists: {TRADE_MATRIX.exists()}")
print(f"Window: {WINDOW_START}-{WINDOW_END}")

## 1. Load & Prepare Import Flows

We need bilateral import flows so that we can identify exactly how much each  
importer gets from the targeted exporter and recompute what remains when that  
exporter is removed.

In [ ]:
# ---- Load raw bilateral trade ----
raw = pd.read_csv(TRADE_MATRIX)
print(f"Loaded {len(raw):,} rows")

# ---- Filter to import quantities in tonnes, positive values only ----
# This matches the filtering in build_quantity_concentration.py.
# Zero-value rows inflate partner counts (see faostat_data_audit.md 2.4).
df = raw[
    (raw["Element"] == "Import quantity")
    & (raw["Unit"] == "t")
    & (raw["Value"] > 0)
].copy()

# ---- Map detailed items to 3 commodity groups ----
df["Commodity"] = df["Item"].map(ITEM_MAP)
df = df[df["Commodity"].notna()].copy()

# ---- Restrict to the analysis window ----
df = df[df["Year"].between(WINDOW_START, WINDOW_END)].copy()

print(f"Filtered to staple imports {WINDOW_START}-{WINDOW_END}: {len(df):,} rows")
print(f"Commodities: {df['Commodity'].value_counts().to_dict()}")

In [ ]:
# ---- Aggregate flows: sum across detailed items and average across years ----
# We first sum within each year (e.g. "Wheat" + "Wheat and meslin flour")
# then take the 3-year mean. This gives a stable baseline for the simulation.

flows_yearly = (
    df.groupby(
        ["Reporter Country Code", "Reporter Countries",
         "Partner Country Code", "Partner Countries",
         "Commodity", "Year"],
        as_index=False,
    )["Value"]
    .sum()
    .rename(columns={"Value": "import_t"})
)

# 3-year mean per bilateral pair
flows = (
    flows_yearly.groupby(
        ["Reporter Country Code", "Reporter Countries",
         "Partner Country Code", "Partner Countries",
         "Commodity"],
        as_index=False,
    )["import_t"]
    .mean()
    .rename(columns={"import_t": "mean_import_t"})
)

# ---- Total imports per importer-commodity (baseline supply) ----
totals = (
    flows.groupby(
        ["Reporter Country Code", "Reporter Countries", "Commodity"],
        as_index=False,
    )["mean_import_t"]
    .sum()
    .rename(columns={"mean_import_t": "baseline_total_import_t"})
)

print(f"Bilateral flow pairs (3-yr mean): {len(flows):,}")
print(f"Importer-commodity groups: {len(totals):,}")

## 2. Targeted Removal — Core Simulation

For each commodity, remove the targeted exporter and compute:  
- **Volume lost** (tonnes from that exporter)  
- **Share lost** (what fraction of the importer's total came from them)  
- **Remaining supply** and **remaining suppliers**  
- Whether the importer loses >50%, >25%, or >10% of imports

In [ ]:
def simulate_removal(flows_df, totals_df, commodity, exporter_name):
    """
    Simulate the removal of a single exporter from a commodity's trade network.
    
    Parameters
    ----------
    flows_df : pd.DataFrame
        Bilateral flows with columns: Reporter Countries, Partner Countries,
        Commodity, mean_import_t.
    totals_df : pd.DataFrame
        Baseline total imports per importer-commodity.
    commodity : str
        One of 'Wheat', 'Rice', 'Maize'.
    exporter_name : str
        The Partner Countries name to remove (e.g. 'Russian Federation').
    
    Returns
    -------
    pd.DataFrame
        One row per affected importer with volume lost, share lost, etc.
    """
    # Filter to this commodity
    cf = flows_df[flows_df["Commodity"] == commodity].copy()
    ct = totals_df[totals_df["Commodity"] == commodity].copy()
    
    # Volume each importer gets from the targeted exporter
    from_target = cf[cf["Partner Countries"] == exporter_name][
        ["Reporter Country Code", "Reporter Countries", "Commodity", "mean_import_t"]
    ].rename(columns={"mean_import_t": "volume_from_target_t"})
    
    if from_target.empty:
        print(f"  WARNING: No imports from {exporter_name} found for {commodity}")
        return pd.DataFrame()
    
    # Merge with baseline totals
    result = ct.merge(
        from_target,
        on=["Reporter Country Code", "Reporter Countries", "Commodity"],
        how="inner",  # only importers who actually import from the target
    )
    
    # Compute impact metrics
    result["share_lost"] = (
        result["volume_from_target_t"] / result["baseline_total_import_t"]
    )
    result["remaining_import_t"] = (
        result["baseline_total_import_t"] - result["volume_from_target_t"]
    )
    
    # Count remaining suppliers after removal
    remaining_partners = (
        cf[cf["Partner Countries"] != exporter_name]
        .groupby(["Reporter Country Code", "Commodity"], as_index=False)
        .agg(remaining_suppliers=("Partner Country Code", "nunique"))
    )
    result = result.merge(
        remaining_partners,
        on=["Reporter Country Code", "Commodity"],
        how="left",
    )
    result["remaining_suppliers"] = result["remaining_suppliers"].fillna(0).astype(int)
    
    # Severity flags
    result["loses_over_50pct"] = result["share_lost"] > 0.50
    result["loses_over_25pct"] = result["share_lost"] > 0.25
    result["loses_over_10pct"] = result["share_lost"] > 0.10
    
    result["removed_exporter"] = exporter_name
    
    return result.sort_values("share_lost", ascending=False)


# ---- Run targeted removal for each commodity ----
targeted_results = []

for commodity, exporter in TARGETED_EXPORTERS.items():
    print(f"\nSimulating removal of {exporter} from {commodity}...")
    result = simulate_removal(flows, totals, commodity, exporter)
    if not result.empty:
        targeted_results.append(result)
        n_affected = len(result)
        n_50 = result["loses_over_50pct"].sum()
        n_25 = result["loses_over_25pct"].sum()
        median_loss = result["share_lost"].median()
        print(f"  Affected importers: {n_affected}")
        print(f"  Lose >50% of imports: {n_50}")
        print(f"  Lose >25% of imports: {n_25}")
        print(f"  Median share lost: {median_loss:.1%}")

targeted = pd.concat(targeted_results, ignore_index=True)
print(f"\nTotal targeted removal rows: {len(targeted):,}")

In [ ]:
# ---- Headline tables per commodity ----
# These are the tables that go into the presentation.

display_cols = [
    "Reporter Countries", "removed_exporter",
    "baseline_total_import_t", "volume_from_target_t",
    "share_lost", "remaining_import_t", "remaining_suppliers",
]

for commodity, exporter in TARGETED_EXPORTERS.items():
    sub = targeted[
        (targeted["Commodity"] == commodity)
    ].copy()
    
    # Show the most-affected importers
    top_affected = sub.head(15)
    
    print(f"\n{'='*70}")
    print(f"{commodity.upper()}: If {exporter} stopped exporting")
    print(f"{'='*70}")
    print(f"Countries losing >50% of imports: {sub['loses_over_50pct'].sum()}")
    print(f"Countries losing >25% of imports: {sub['loses_over_25pct'].sum()}")
    print(f"Countries losing >10% of imports: {sub['loses_over_10pct'].sum()}")
    print(f"Total volume removed: {sub['volume_from_target_t'].sum():,.0f} tonnes")
    print(f"\nTop 15 most-affected importers:")
    print(
        top_affected[display_cols].to_string(
            index=False,
            float_format="{:,.0f}".format,
            formatters={"share_lost": "{:.1%}".format},
        )
    )

## 3. Random Removal Baseline

To demonstrate that targeted removal is *structurally* more damaging than  
losing an arbitrary supplier, we run the same simulation with randomly chosen  
exporters. If the median impact of targeted removal exceeds the 95th  
percentile of random removal, the concentration finding is robust.

In [ ]:
rng = np.random.default_rng(RNG_SEED)

random_comparison = []

for commodity, target_exporter in TARGETED_EXPORTERS.items():
    print(f"\n{commodity}: Running {N_RANDOM_TRIALS} random removal trials...")
    
    # Get all exporters for this commodity (excluding the targeted one)
    cf = flows[flows["Commodity"] == commodity]
    all_exporters = cf["Partner Countries"].unique()
    other_exporters = [e for e in all_exporters if e != target_exporter]
    
    # Targeted removal impact (the real statistic we're comparing against)
    targeted_sub = targeted[targeted["Commodity"] == commodity]
    targeted_median_loss = targeted_sub["share_lost"].median()
    targeted_n_over50 = targeted_sub["loses_over_50pct"].sum()
    
    # Run random trials
    trial_median_losses = []
    trial_n_over50 = []
    
    for trial in range(N_RANDOM_TRIALS):
        # Pick a random exporter
        random_exporter = rng.choice(other_exporters)
        result = simulate_removal(flows, totals, commodity, random_exporter)
        
        if result.empty:
            trial_median_losses.append(0.0)
            trial_n_over50.append(0)
        else:
            trial_median_losses.append(result["share_lost"].median())
            trial_n_over50.append(result["loses_over_50pct"].sum())
    
    # Compare
    p95_random = np.percentile(trial_median_losses, 95)
    mean_random = np.mean(trial_median_losses)
    
    random_comparison.append({
        "commodity": commodity,
        "targeted_exporter": target_exporter,
        "targeted_median_loss": targeted_median_loss,
        "targeted_n_over50": targeted_n_over50,
        "random_mean_median_loss": mean_random,
        "random_p95_median_loss": p95_random,
        "random_mean_n_over50": np.mean(trial_n_over50),
        "targeted_exceeds_p95": targeted_median_loss > p95_random,
    })
    
    print(f"  Targeted ({target_exporter}): median loss = {targeted_median_loss:.1%}")
    print(f"  Random mean: median loss = {mean_random:.1%}")
    print(f"  Random 95th percentile: {p95_random:.1%}")
    print(f"  Targeted exceeds P95 of random: {targeted_median_loss > p95_random}")

comparison_df = pd.DataFrame(random_comparison)
print("\n" + comparison_df.to_string(index=False))

In [ ]:
# ---- Visualization: targeted vs random removal ----
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, row in zip(axes, comparison_df.itertuples()):
    commodity = row.commodity
    exporter = row.targeted_exporter
    
    # Get random trial data for this commodity (re-run is fast with seed)
    cf = flows[flows["Commodity"] == commodity]
    other_exporters = [
        e for e in cf["Partner Countries"].unique()
        if e != exporter
    ]
    
    trial_rng = np.random.default_rng(RNG_SEED)
    trial_losses = []
    for _ in range(N_RANDOM_TRIALS):
        re = trial_rng.choice(other_exporters)
        r = simulate_removal(flows, totals, commodity, re)
        trial_losses.append(r["share_lost"].median() if not r.empty else 0.0)
    
    # Histogram of random trial results
    ax.hist(trial_losses, bins=30, alpha=0.7, edgecolor="white", label="Random removal")
    
    # Vertical line for targeted removal
    ax.axvline(
        row.targeted_median_loss, color="red", linewidth=2.5,
        linestyle="--", label=f"Remove {exporter.split()[0]}"
    )
    
    ax.set_xlabel("Median share lost per importer")
    ax.set_title(f"{commodity}")
    ax.legend(fontsize=8)

axes[0].set_ylabel("Number of random trials")
fig.suptitle(
    "Targeted vs Random Supplier Removal: Median Import Share Lost",
    fontsize=14, y=1.02,
)
plt.tight_layout()
plt.savefig(str(VIZ_DIR / "targeted_vs_random_removal.png"),
            dpi=150, bbox_inches="tight")
plt.show()

## 4. Multi-Exporter Cascade

What happens when the **top-2** or **top-3** exporters are removed simultaneously?  
This models a correlated shock (e.g. Black Sea disruption affecting both Russia  
and Ukraine, or a simultaneous drought across major grain belts).

In [ ]:
def simulate_multi_removal(flows_df, totals_df, commodity, exporters_to_remove):
    """
    Simulate the simultaneous removal of multiple exporters.
    
    Parameters
    ----------
    flows_df : pd.DataFrame
        Bilateral flows.
    totals_df : pd.DataFrame
        Baseline totals.
    commodity : str
        Commodity name.
    exporters_to_remove : list[str]
        List of Partner Countries names to remove simultaneously.
    
    Returns
    -------
    pd.DataFrame
        Impact per importer.
    """
    cf = flows_df[flows_df["Commodity"] == commodity].copy()
    ct = totals_df[totals_df["Commodity"] == commodity].copy()
    
    # Sum volume from ALL targeted exporters per importer
    from_targets = (
        cf[cf["Partner Countries"].isin(exporters_to_remove)]
        .groupby(
            ["Reporter Country Code", "Reporter Countries", "Commodity"],
            as_index=False,
        )["mean_import_t"]
        .sum()
        .rename(columns={"mean_import_t": "volume_from_targets_t"})
    )
    
    if from_targets.empty:
        return pd.DataFrame()
    
    result = ct.merge(
        from_targets,
        on=["Reporter Country Code", "Reporter Countries", "Commodity"],
        how="inner",
    )
    
    result["share_lost"] = (
        result["volume_from_targets_t"] / result["baseline_total_import_t"]
    )
    result["remaining_import_t"] = (
        result["baseline_total_import_t"] - result["volume_from_targets_t"]
    )
    
    # Remaining suppliers
    remaining = (
        cf[~cf["Partner Countries"].isin(exporters_to_remove)]
        .groupby(["Reporter Country Code", "Commodity"], as_index=False)
        .agg(remaining_suppliers=("Partner Country Code", "nunique"))
    )
    result = result.merge(
        remaining,
        on=["Reporter Country Code", "Commodity"],
        how="left",
    )
    result["remaining_suppliers"] = result["remaining_suppliers"].fillna(0).astype(int)
    result["loses_over_50pct"] = result["share_lost"] > 0.50
    result["loses_over_25pct"] = result["share_lost"] > 0.25
    result["n_exporters_removed"] = len(exporters_to_remove)
    result["removed_exporters"] = ", ".join(exporters_to_remove)
    
    return result.sort_values("share_lost", ascending=False)


# ---- Run cascade for each commodity ----
cascade_results = []

for commodity, exporters in CASCADE_EXPORTERS.items():
    print(f"\n{'='*60}")
    print(f"{commodity.upper()} CASCADE")
    print(f"{'='*60}")
    
    # Remove top-1, top-2, top-3 sequentially
    for n_remove in [1, 2, 3]:
        to_remove = exporters[:n_remove]
        result = simulate_multi_removal(flows, totals, commodity, to_remove)
        
        if not result.empty:
            cascade_results.append(result)
            n_50 = result["loses_over_50pct"].sum()
            n_25 = result["loses_over_25pct"].sum()
            total_lost = result["volume_from_targets_t"].sum()
            
            print(f"\n  Remove top-{n_remove}: {', '.join(to_remove)}")
            print(f"  Countries losing >50%: {n_50}")
            print(f"  Countries losing >25%: {n_25}")
            print(f"  Total volume removed: {total_lost:,.0f} t")

cascade = pd.concat(cascade_results, ignore_index=True)

In [ ]:
# ---- Cascade visualization: bar chart of countries losing >50% ----
cascade_summary = (
    cascade.groupby(["Commodity", "n_exporters_removed", "removed_exporters"],
                    as_index=False)
    .agg(
        countries_over_50pct=("loses_over_50pct", "sum"),
        countries_over_25pct=("loses_over_25pct", "sum"),
        total_volume_lost_t=("volume_from_targets_t", "sum"),
    )
)

fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(cascade_summary))
width = 0.35

bars1 = ax.bar(
    x - width / 2, cascade_summary["countries_over_50pct"],
    width, label="Lose >50% of imports", color="#d32f2f",
)
bars2 = ax.bar(
    x + width / 2, cascade_summary["countries_over_25pct"],
    width, label="Lose >25% of imports", color="#ff9800",
)

# Labels
labels = [
    f"{row.Commodity}\nRemove top-{row.n_exporters_removed}"
    for row in cascade_summary.itertuples()
]
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=9)
ax.set_ylabel("Number of importing countries affected")
ax.set_title("Cascade Simulation: Impact of Removing Top Exporters")
ax.legend()

# Value labels on bars
for bar in bars1:
    h = bar.get_height()
    if h > 0:
        ax.text(bar.get_x() + bar.get_width() / 2, h + 0.3,
                str(int(h)), ha="center", va="bottom", fontsize=9)
for bar in bars2:
    h = bar.get_height()
    if h > 0:
        ax.text(bar.get_x() + bar.get_width() / 2, h + 0.3,
                str(int(h)), ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.savefig(str(VIZ_DIR / "cascade_removal_impact.png"),
            dpi=150, bbox_inches="tight")
plt.show()

## 5. Most Vulnerable Countries — Summary Table

A single table showing every country that would lose >25% of any staple  
from the targeted removal. This is the table that goes in the final deck.

In [ ]:
# ---- Build the headline vulnerability table ----
vulnerable = targeted[targeted["loses_over_25pct"]].copy()

# Convert to thousands of tonnes for readability
vulnerable["baseline_kt"] = vulnerable["baseline_total_import_t"] / 1000
vulnerable["lost_kt"] = vulnerable["volume_from_target_t"] / 1000

headline = vulnerable[
    ["Reporter Countries", "Commodity", "removed_exporter",
     "baseline_kt", "lost_kt", "share_lost", "remaining_suppliers"]
].sort_values(["Commodity", "share_lost"], ascending=[True, False])

print(f"Countries losing >25% of a staple import ({WINDOW_START}-{WINDOW_END} mean):")
print(f"Total entries: {len(headline)}\n")
print(
    headline.to_string(
        index=False,
        float_format="{:,.1f}".format,
        formatters={"share_lost": "{:.1%}".format},
    )
)

## 6. Save Outputs

In [ ]:
# ---- Save the full targeted-removal results ----
# This is the detailed output — every importer, every commodity.
targeted_out = OUTPUT_DIR / "supplier_removal_simulation.csv"
targeted.to_csv(targeted_out, index=False)
print(f"Saved targeted removal results: {targeted_out}")
print(f"  Rows: {len(targeted):,}")

# ---- Save the cascade results ----
cascade_out = OUTPUT_DIR / "supplier_removal_cascade.csv"
cascade.to_csv(cascade_out, index=False)
print(f"Saved cascade results: {cascade_out}")
print(f"  Rows: {len(cascade):,}")

# ---- Save the comparison summary ----
comparison_out = OUTPUT_DIR / "targeted_vs_random_comparison.csv"
comparison_df.to_csv(comparison_out, index=False)
print(f"Saved comparison summary: {comparison_out}")

# ---- Print headline numbers for the presentation ----
print("\n" + "=" * 60)
print("HEADLINE NUMBERS FOR THE PRESENTATION")
print("=" * 60)
for commodity, exporter in TARGETED_EXPORTERS.items():
    sub = targeted[targeted["Commodity"] == commodity]
    n_50 = sub["loses_over_50pct"].sum()
    n_25 = sub["loses_over_25pct"].sum()
    total_vol = sub["volume_from_target_t"].sum() / 1e6
    worst_country = sub.iloc[0]["Reporter Countries"]
    worst_share = sub.iloc[0]["share_lost"]
    print(f"\n  {commodity} (remove {exporter}):")
    print(f"    {n_50} countries lose >50% of {commodity.lower()} imports")
    print(f"    {n_25} countries lose >25%")
    print(f"    {total_vol:.1f} Mt removed from global trade")
    print(f"    Most exposed: {worst_country} ({worst_share:.0%})")

print("\nVisualizations saved to: visualizations/")
print("  - targeted_vs_random_removal.png")
print("  - cascade_removal_impact.png")